# MU direction classification

## Data and protocol

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, SVG

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config_ml.json').exists())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import read_config, download_data
from src.ml import FEATURES, make_dataset, make_blocks
from src.ml_pipeline import verify_protocol, run_ml_pipeline

verify_protocol(ROOT)
config = json.loads((ROOT / 'config_ml.json').read_text())
assert (ROOT / 'reports/manifest.json').exists(), '先运行 V1 或恢复原始私有快照'
data = download_data(ROOT, read_config(ROOT), refresh=False)
print('ML001:', config['ticker'], '| 模型阈值:', config['probability_threshold'])

## Features and next-open labels

In [ ]:
dataset = make_dataset(data['MU'], horizon=config['horizon_sessions'])
blocks, split_audit, purged_rows = make_blocks(dataset, config)
example_columns = ['entry_date', 'label_end_date', 'return_1', 'momentum_20', 'rsi_14', 'forward_return', 'target']
display(blocks['train'][example_columns].head().round(4))
display(dataset[['entry_date', 'label_end_date', 'forward_return', 'target']].tail(3))
print('模型的 X 只有：', FEATURES)

## Chronological splits and label purging

In [ ]:
display(split_audit)
display(purged_rows)
assert blocks['train'].label_end_date.max() < blocks['validation'].index.min()
assert blocks['refit'].label_end_date.max() < blocks['test'].index.min()
print('跨界标签已隔离；训练和评估日期通过检查。')

## Fixed classifiers and baseline

In [ ]:
study = run_ml_pipeline(ROOT)
display(study['fit_audit'])
scaler = study['fits']['test']['LogisticRegression']['scale']
np.testing.assert_allclose(scaler.mean_, study['blocks']['refit'][FEATURES].mean())
print('最终 scaler 的均值只来自 refit 训练块。')

## Classification metrics

In [ ]:
columns = ['block', 'model', 'observations', 'positive_rate', 'predicted_positive_rate',
           'accuracy', 'balanced_accuracy', 'roc_auc', 'brier_score', 'log_loss']
display(study['classification_metrics'][columns].round(4))
display(SVG(filename=str(ROOT / 'figures/ml_classification.svg')))

## Trading after transaction costs

In [ ]:
trade_columns = ['block', 'ticker', 'strategy', 'entry_start', 'end', 'cumulative_return',
                 'annualized_volatility', 'maximum_drawdown', 'sharpe_rf0',
                 'session_exposure', 'closed_trades', 'win_rate_closed_trades']
display(study['trading_metrics'][trade_columns].round(4))
display(SVG(filename=str(ROOT / 'figures/ml_equity_drawdown.svg')))

## Cost and calendar-year sensitivity

In [ ]:
costs = study['cost_sensitivity']
display(costs.loc[(costs.block == 'test') & costs.strategy.isin(['LogisticRegression', 'RandomForest']),
                  ['strategy', 'one_way_cost_bps', 'cumulative_return', 'maximum_drawdown', 'one_way_transactions']].round(4))
display(study['yearly_classification'][['feature_year', 'model', 'observations', 'accuracy', 'balanced_accuracy']].round(4))
display(study['yearly_account_returns'].round(4))